# Working with Numpy Arrays

In this notebook we'll cover reading data from a file into a Numpy array, separating out different parts of the array to different namespaces and reshaping some of the elements. Additionally, we'll cover using some of the basic statistical functions and methods associated with float and integer arrays and be able to find where maximum and minimum values occur within an array.

## Reading in data with built-in Python functions
It is difficult to easily talk about reading data from a file into a Python program simply because there are so many ways to do so! Depending on what the data is and what you want to do with it, there will be a number of different methods to read the file in.
* `open(file)` is a built-in function of Python
* The `csv` module is great for reading and writing standard CSV and Excel formatted data.
    * `csv.reader(file)` is the standard function in this module.
    * `csv.DictReader(file)` converts the table to a dictionary, using the column headers as the dictionary keys
* `np.loadtxt(file)` stores data as a numpy array.
* `pd.read_csv(file)` also reads CSV files, storing it as a DataFrame. Pandas is another module built on NumPy that makes it easy to work with tables.

Starting with the basics, if you wanted to read in a file with just data, you can use the `open()` command. Make sure you clear your memory by closing the file when you're done.

In [ ]:
# Get data for the Relative Oceanic Nino Index (RONI)
file = open("/home/jovyan/pragmatic_python_for_weather/guides/data/roni_values.dat", "r")
content = file.read()
file.close()
#print(content)

I gave the function the full path, or absolute path. Starting from the root directory, what are all the directories I have to open to get to this file? The `/` at the very beginning indicates this is a full path.

We can also provide the relative filename from the directory we are currently in. To open this file, we went to the guides directory. So the relative path is just `./data/nao_table.dat`. The `./` at the beginning says this is from the current working directory. We can also just put in `data/nao_table.dat` without either the leading `/` or `./` syntax.

When using a relative path, `../` is how you go up a level. Since we're in `/home/jovyan/pragmatic_python_for_weather/guides/`, using `../syntax_problems/data` would allow us to access the data files we have for syntax problems.

In [ ]:
# Open the file in read mode
file = open("./data/roni_values.dat", "r")

# Read the entire content of the file
content = file.read()

# Close the file
file.close()

#print(content)

We can both shorten the code and make it more readable by opening the file using the `with` command. With the file open, read and print the content. When you break out of the indention, the file will close without you commanding it to do so in an extra line.

In [ ]:
# Use with to automatically close the file
with open("./data/roni_values.dat", "r") as file:
    content = file.read()
#print(content)

The above examples involved the `.read()` function, which reads the whole file and stores it as a single variable. Instead, we can use the `.readline()` function to read files line by line.

In [ ]:
with open("./data/roni_values.dat", "r") as file:
    line = file.readline()
    while line:
        print(line.strip())
        line = file.readline()

Unfortunately, you have to work with that line while you have it. Once you go onto the next line, the code hasn't stored the previous line. You would have to manually save them into a list.

To make this easy, `.readlines()` reads each line and stores them as a list of lines.

Printing the lines, notice they all contain the `\n` newline character. Sometimes, you may also see the related `/r` character in binary files. To get rid of these characters, we need to use the `.rstrip()` function on each line individually.

In [ ]:
with open("./data/roni_values.dat", "r") as file:
    lines = file.readlines()
    #lines = [line.rstrip() for line in file.readlines()]
#print(lines)

# CSV module
If a file is organized as a comma-separated version (CSV) file, then data can be read in with the `csv` module.

In [ ]:
import csv

with open("./data/roni_table.csv", newline='') as csvfile:
    roni_values = csv.reader(csvfile, delimiter=',')
    for row in roni_values:
        print(row)
        #print(', '.join(row))

# Change the code above to print just the values for September from each year.

As above, can use the `list()` to convert the csv reader object to a list of each row. This works out to be a list of lists, also called a 2D list.

In [ ]:
import csv

with open("./data/roni_table.csv", newline='') as csvfile:
    roni_values = csv.reader(csvfile)
    roni_years = list(roni_values)

# Print the last 5 years of data

# Make a loop so each line is on its own row.

A particularly useful function in the `csv` module is the `csv.DictReader()` function. This reads CSV files and converts the data into a dictionary. It uses the header row to assign the keys and all other rows as the values.

In [ ]:
with open("./data/roni_table.csv", newline='') as csvfile:
    roni_values = csv.DictReader(csvfile)

    # Generate a list of dictionaries
    roni_lists_of_dictionaries = list(roni_values)

# Convert to a dictionary of lists
roni_dictionaries_of_lists = {keys: [row[keys] for row in roni_lists_of_dictionaries] for keys in roni_lists_of_dictionaries[0]}

# Print just the list for September.


## Reading in data with Numpy
Specific to the Numpy module, there are simple read functions, and we are going to use one here. In the future we will use some read functionality from a package called Pandas (https://pandas.pydata.org) that is in some ways built on top of the Numpy module and brings with it a whole lot of additional functionality. In this example we'll read in an ASCII (plain text) file. We can then use the data to calculate or graph stuff to answer scientific questions! Specifically, the data that we are reading in is the North Atlantic Oscillation (NAO) data from 1950 to 2024.

Source: https://www.cpc.ncep.noaa.gov/products/precip/CWlink/pna/nao.shtml

In [ ]:
import numpy as np

To read in a file, we'll use the `loadtxt()` function from the numpy module.

How can I find out more about this function? Well, there are many ways, but two of the easiest would be to 
1) google <span style="font-family:Courier"> **numpy loadtxt** </span> or
2) in a new cell below type ```np.loadtxt?```.
3) click on the function in the code cell and press `Shift+Tab`

The second way will bring up the manual pages from the numpy module to a dialogue box at the bottom of your web browser. You can also put your cursor in the function name and press shift+tab to get a dialogue box that contains the documentation for that particular function.

In [ ]:
# Get data for the Relative Oceanic Nino Index (RONI)
roni_data = np.loadtxt("data/roni_data.txt")

This function loaded the data just fine. But if we open the file, we see there are just rows and columns of data. There's no context provided, so we have to get it from wherever we found the data.

If we output the data, we see a 2D array.

In [ ]:
# What does the data look like?
print(roni_data)

Sometimes, the file has metadata - data about data. Simple things like a label for what the data is. The units might be specified. Sometimes there may be information about where or how the data was collected.

Metadata are typically given as a header row. So `np.loadtxt` provides a `skiprows` keyword argument to specify how many rows to skip before we get to data. That way we skip over our metadata.

It also provides keywords like `delimiter` to specify whether the columns are separated by commas, tabs, pipes `|`, or something else.

In [ ]:
# Changing the filename, we find the same data but with header rows.
# It's also organized differently.
roni_list = np.loadtxt("data/roni_table.dat", skiprows=2)

Notice the first column contains dates, but were read in just like any other column of data. We'll want to take care of that. Let's do some investigating.

In [ ]:
# What is the shape of the 2D arrays that we have read in? Use variable.shape


## Separating Columns
With multidimensional arrays, we can isolate different columns and rows into separate arrays. This might be advantageous for working with the data where you can ignore certain parts of the data during your analysis. Separating dimensions may also allow you to work with them in different ways.

Remember, we can subset via `array_variable[row,column]`

In [ ]:
# Separate the year column of the NAO data from the rest of the data.
# Get the NAO data in its own array, separated from the years column.
roni_values = 

# Check the shape of the nao_values and check the values to assure the dates are excluded.


In [ ]:
# Now get just the years column in its own array.
# Again, print the shape and the array itself.
roni_years = 

# Check the type of data for both arrays. Can change using array.astype().

Create a new array called `months` using a list of the months. Remember each month has to be its own string.

In [ ]:
# Create an array of month values.
months = 

In [ ]:
# Subset the array so you only have data from 1997.
year = 1997

# Manually: look for index 47
roni_year = roni_values[]

# Programmatically with a calculation: year - 1950
roni_year = roni_values[]

# Using numpy syntax to search: years==year
roni_year = roni_values[]


In [ ]:
# Slice to get one decade of nao data
years_1950s = 

## Reshape
We've got a 2d array of data, but there is more inherent organization to this data that might be better to represent as a table instead. We can reshape an array by our known structure.


In [ ]:
# Find three decades of data, typical for climatological baselines.
# We're currently using 1990-2019
roni_three_decades = 

# Reshape the data so you have 3 rows, each with a decade of monthly data.
# Let the number of rows be as many as they need to be with `variable.reshape(-1,...)`


## Common Functions in Numpy
Hey, you're training to be a scientist! Do you ever compute averages? maximum? minimum? Numpy can help!

Statistical Functions: http://docs.scipy.org/doc/numpy/reference/routines.statistics.html
* `max`
* `min`
* `std`
* `median`
* `mean`
* `average`
* `nanmean`
* `corrcoef`

A number of these functions can also be called as methods on a Numpy data object.

```python
jan_avg_function = np.mean(data[::12, 1])
jan_avg_method = data[::12, 1].mean()
print(jan_avg)
print(jan_avg_method)
```
Output:
```
0.12416666666666669
0.12416666666666669
```
Note: The `average` function actually computes a weighted average. The default weight is 1 for all observations, but this function gives you the option of changing them. The `mean` function just computes the straight arithmatic mean of the array along an axis. It does not have the ability to provide weights.

Each of these Numpy functions can be computed over the entire array and return a single value, or over a particular axis to return the average for each row or column depending on which axis was chosen. To do this you would specify the keyword argument of `axis` and set it to the desired axis number (e.g., 0, 1, 2, etc.)

In [ ]:
# Let's keep using roni_years, months, roni_three_decades

# Compute the average value of NAO for the month of January over these decades.
# Which column or row is January? How much of that column or row do we want?
jan_avg = 

# Now find the standard deviation
jan_stdev = 

In [ ]:
# June Average NAO for the three decades
jun_stdev = 

In [ ]:
# Average and standard deviation from the whole period?
roni_climo_average = 
roni_climo_stdev = 

What if we want an array with the average value for each of the 12 months? Or an array with the average value for each year in the dataset?

Numpy helps us out! When we run the `.mean()` method with nothing in the parentheses, we get a single value returned like we've been doing. But the method allows us to put `axis=0` or `axis=1` in the parentheses to make these arrays. So the rows (months) are the 0th axis and the columns (years) are the 1st axis.

In [ ]:
# Average and standard deviation for each month
roni_month_averages = 

roni_month_stdevs = 

# Advanced: Average of June-July-August range for all years.

In [ ]:
# Now let's compute the max RONI value from the whole dataset
max_roni_value = 

In [ ]:
# Minimum NAO value from the whole dataset
min_roni_value = 

## Finding Values in an Array

Sometimes you want an efficient way to search an array for a particular value or values. The `np.where` function can use logical operations to identify indices that correspond to where the logical condition is True. The returned value is a tuple of arrays, so it will require a bit to get out individual index values.

```python
a = np.array([-2, -1, 0, 1, 2])
print(np.where(a <= -1))
```
Output:
```
(array([0, 1],))
```
We can also look for a specific value. Let's do this with a 2D array.
```python
a = np.array([[2000,2001,2002,2003,2004],
             [-2, -1, 0, 1, 2]])
print(np.where(a == 1))
```
Output:
```
(array([1]), array([3]))
```
Notice the outputs are tuples with arrays in them. 
* In our 1D array, only one array is in the tuple, and it says the 0th and 1st index are less than or equal to one.
* In our 2D array, two arrays are in the tuple corresponding to the rows and columns. The 1st row and 3rd index is the only place where the value 1 occurs.

With two arrays in the tuple, we can unpack the output by capturing the values in two variables.
```python
row, col = np.where(a == 1)
```
Output:
```


In [ ]:
# So when did the maximum value occur?
# Save the year and month index of the max RONI value.
# Print the max value, year, and month number, all in floats or ints.


## Exercise 1
Find the maximum and minimum RONI that occurs between 1970-1999 and when it occurs.

In [ ]:
# Find the maximum and minimum that occurs between 1970 - 1999 and where it occurs


## Exercise 2
Use the where function to find all of the elements where one of the following conditions are true:  
https://numpy.org/doc/stable/reference/generated/numpy.where.html  

NAO >= 1.25

NAO <= -1.25

Note: The `and` condition will not work in this case. To assess the combination of two inequalities with Numpy arrays we need to exploit the mathematical operations related to boolean arrays.

`True * True = True`

`True * False = False`

`False * False = False`

`True + True = True`

`True + False = True`

`False + False = False`

In [ ]:
# Multiple Conditions
# Find all index values where RONI is >= 1.25 or <= -1.25

# Print the indices, the values, the years, and the months

# Store these in separate variables and print using the zip function
for roni, yr, mo in zip(extreme_roni,extreme_roni_yr,extreme_roni_mo):
    print(f'{roni:>5.2f}   {yr}   {mo}')